In [1]:
import sys
sys.path.append("..")

In [2]:
from openai import AsyncOpenAI
from src.embeddings import EmbeddingModel
from sentence_transformers import CrossEncoder
from src.retrieval import Reterival
from src.prompt_template import build_llm_request, ensure_article_citations
from ragas import EvaluationDataset
from ragas import evaluate
from src.config import Config
from src.schemas import Payload
from src.reranker import Reranker
from src.llm.factory import get_llm
from ragas.llms import llm_factory
from src.api.versions.v1.rag import get_payload_from_points 
from ragas.metrics import LLMContextRecall, Faithfulness, FactualCorrectness
import json
import tqdm
from langdetect import detect
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

c:\Users\Lenovo\Learning\MLOPS_MENA\projects\ArabicLegalDocumentProject\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\Lenovo\AppData\Local\Temp\ipykernel_7064\3471228325.py:14: DeprecationWarning: Importing LLMContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import LLMContextRecall
  from ragas.metrics import LLMContextRecall, Faithfulness, FactualCorrectness
C:\Users\Lenovo\AppData\Local\Temp\ipykernel_7064\3471228325.py:14: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import LLMContextRecall, 

In [3]:
ar_embedding = EmbeddingModel(model_path=Config().ARABIC_EMBEDDING,
                                                        lang='arabic')
en_embedding = EmbeddingModel(model_path=Config().ENGLISH_EMBEDDING , 
                              lang='english')

In [4]:
reteriver = Reterival(
    arabic_embedding=ar_embedding,
    english_embedding=en_embedding,
    
)

In [5]:
llm = get_llm(config = Config())

In [6]:
rerank_model = CrossEncoder(Config().RERANKER_MODEL)
reranker = Reranker(rerank_model)

c:\Users\Lenovo\Learning\MLOPS_MENA\projects\ArabicLegalDocumentProject\.venv\Lib\site-packages\qdrant_client\async_qdrant_remote.py:233: UserWarning: Failed to obtain server version. Unable to check client-server compatibility. Set check_compatibility=False to skip version check.
  show_warning(
Default prompt name is set to 'query'. This prompt will be applied to all inference calls, except if a `prompt` or `prompt_name` parameter is provided.


In [8]:
with open("../evaluation/data/civil_code_rag_qa_benchmark.json", encoding="utf-8") as f:
    data = json.load(f)

In [7]:
lang_map = {"ar":"arabic", "en":"english"}

In [22]:
dataset =[]

for record in tqdm.tqdm(data):
    try:
        user_input = record['user_input']
        refrence = record['response']
        lang = lang_map[detect(user_input)]
        points = await  reteriver.search(
            collection_name=Config().COLLECTION_NAME ,
            text = user_input,
            lang=lang,
            reterival_limit = Config().RETERIVAL_LIMIT,
            reterival_renkad_limit = Config().RETERIVAL_RRF_LIMIT
        )
        retrieved_contexts = [point.payload['ar_text' if lang == 'arabic' else "text_en"] for point in points ]

        payloads = get_payload_from_points(points)
        
        reanked_payloads = reranker.rerank(
                query=user_input,
                payloads=payloads,
                lang=lang,
                limit=Config().RETERIVAL_RERANKED_LIMIT,
            )
        
        llm_request = build_llm_request(
            query=user_input,
            payloads=reanked_payloads,
            language=lang,
            max_docs=Config().RETERIVAL_RERANKED_LIMIT,
            max_chars=24000,
        )
        
        response = await llm.generate(llm_request)
        
        answer = response.answer
        
        dataset.append(
            {
                "user_input":user_input,
                "retrieved_contexts":retrieved_contexts,
                "response":answer,
                "reference":refrence
            }
        )
    except Exception as e:
        
        print(e)
        

100%|██████████| 86/86 [42:04<00:00, 29.35s/it]


In [23]:
evaluation_dataset = EvaluationDataset.from_list(data=dataset)

In [24]:
evaluation_df = evaluation_dataset.to_pandas()
evaluation_df.head()

,user_input,retrieved_contexts,response,reference
0,What is the legal age of majority?,[A person who has reached the age of discretio...,"Based on the provided documents, the legal age...",The age of majority is twenty-one years comple...
1,ما هو سن الرشد القانوني؟,[، كل شخص بلغ سن الرشد متمتعا بقواه العقلية، و...,بناءً على السياق القانوني المعطى، فإن سن الرشد...,سن الرشد هو إحدى وعشرون سنة ميلادية كاملة.
2,When does a person have full legal capacity to...,[All persons attaining majority in possession ...,"Based on the provided documents, a person has ...","A person who has attained majority, possesses ..."
3,متى يكون الشخص كامل الأهلية لمباشرة حقوقه المد...,[، كل شخص بلغ سن الرشد متمتعا بقواه العقلية، و...,بناءً على السياق القانوني المطروح، فإن الشخص ي...,يكون الشخص كامل الأهلية لمباشرة حقوقه المدنية ...
4,Who is considered incapable of exercising civi...,"[A person devoid of discretion, owing to youth...","Based on the provided documents, the following...",A person who lacks discretion because of youth...


In [26]:
evaluation_df.to_csv("../evaluation/data/evaluation_results.csv", index=False)

In [8]:
import ast
import pandas as pd
from ragas import EvaluationDataset

evaluation_df = pd.read_csv("../evaluation/data/evaluation_results.csv")
evaluation_df["retrieved_contexts"] = evaluation_df["retrieved_contexts"].apply(ast.literal_eval)

evaluation_dataset = EvaluationDataset.from_list(
    evaluation_df.to_dict(orient="records")
)

In [10]:
evaluation_dataset

EvaluationDataset(features=['user_input', 'retrieved_contexts', 'response', 'reference'], len=86)

In [11]:
client = AsyncOpenAI(base_url="http://localhost:11434/v1", 
                     api_key='not-needed',
                     timeout=120
                     )  # Reads OPENAI_API_KEY from the environment
evaluator_llm = llm_factory(model="gpt-oss:20b-cloud", client=client, max_tokens=32768)



In [12]:
evaluator_llm.model_args

{'temperature': 0.01, 'top_p': 0.1, 'max_tokens': 32768}

In [13]:
from ragas.run_config import RunConfig

result = evaluate(
    dataset=evaluation_dataset,
    metrics=[LLMContextRecall(), Faithfulness(), FactualCorrectness()],
    llm=evaluator_llm,
    run_config=RunConfig(max_workers=1),
)

Evaluating: 100%|██████████| 258/258 [4:45:02<00:00, 66.29s/it]


In [14]:
result

{'context_recall': 0.9353, 'faithfulness': 0.5444, 'factual_correctness(mode=f1)': 0.3504}